In [0]:
fact_inspections = spark.table("fact_inspections")
dim_restaurant = fact_inspections.select(
    "business_name"
).distinct()

In [0]:
dim_restaurant = fact_inspections.select("business_name").distinct()

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_restaurant = dim_restaurant.withColumn(
    "restaurant_id",
    monotonically_increasing_id()
)

In [0]:
dim_restaurant = dim_restaurant.select(
    "restaurant_id",
    "business_name"
)

In [0]:
dim_restaurant.write.mode("overwrite").saveAsTable("dim_restaurant")

In [0]:
fact_inspections_final = fact_inspections.join(
    dim_restaurant,
    on="business_name",
    how="left"
).select(
    "inspection_id",
    "inspection_date",
    "inspection_score",
    "inspection_result",
    "restaurant_id",
    "city",
    "zip_code"
)

In [0]:
fact_inspections_final.write.mode("overwrite").saveAsTable("fact_inspections_final")

# ================================
# SCD TYPE 2 IMPLEMENTATION
# ================================


In [0]:
from pyspark.sql.functions import current_date, lit

# Recreate clean SCD table
dim_restaurant = spark.table("dim_restaurant")

dim_restaurant_scd = dim_restaurant.withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

dim_restaurant_scd.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_restaurant_scd")

spark.table("dim_restaurant_scd").show(5)

In [0]:
from pyspark.sql.functions import col

existing = spark.table("dim_restaurant_scd")

# pick ONE row
row = existing.limit(1).collect()[0]
rid = row["restaurant_id"]

# split dataset
old_row = existing.filter(col("restaurant_id") == rid)
remaining = existing.filter(col("restaurant_id") != rid)

# close old version
closed_old = old_row.withColumn("is_current", lit(False)) \
    .withColumn("end_date", current_date())

# new version
new_version = old_row.withColumn("business_name", lit("UPDATED_RESTAURANT")) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

# combine
final_scd = remaining.unionByName(closed_old).unionByName(new_version)

# save
final_scd.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_restaurant_scd")

spark.table("dim_restaurant_scd").show(10)